In [13]:
# import tonic
# import torch

# to_frame = tonic.transforms.ToFrame(
#     sensor_size=tonic.datasets.NMNIST.sensor_size, time_window=1e3
# )
# test_dataset = tonic.datasets.NMNIST(".", transform=to_frame, train=False)

# # Define dataloader
# batch_size = 32

# data_loader = torch.utils.data.DataLoader(
#     test_dataset,
#     shuffle=True,
#     batch_size=batch_size,
#     collate_fn=tonic.collation.PadTensors(),
# )

In [14]:
# import numpy as np

# def salvar_dataset_npz(loader, arquivo):
#     dados = []
#     labels = []

#     i = 0
#     for batch in loader:

#         if i == 32:
#             break
    
#         i += 1

#         x, y = batch

#         if torch.is_tensor(x):
#             x = x.cpu().numpy()
#             y = y.cpu().numpy()

#         dados.append(x)
#         labels.append(y)

#     dados = [arr[:, :300] for arr in dados]
#     dados = np.concatenate(dados, axis=0)

#     labels = np.concatenate(labels, axis=0)

#     np.savez_compressed(
#         arquivo,
#         data=dados,
#         labels=labels
#     )

#     print(f"Arquivo '{arquivo}.npz' salvo.")

In [15]:
# salvar_dataset_npz(data_loader, "nir_examples/cnn_teste.npz")

# NeuroHls

In [1]:
from neuro_hls import *

In [2]:
neuro_hls = NeuroHls("z_test_18_mar")

## Definindo a Implementação do Modelo

In [18]:
nir_file = "nir_examples/cnn_sinabs.nir"

In [19]:
model = neuro_hls.read_nir_file(nir_file)

In [20]:
print(model)

-------------------------------------------------------
Input ([ 2 34 34]) - layer name: 'input'
-------------------------------------------------------
	Is recurrent: NO
	Dependencies:

-------------------------------------------------------
Conv2d (input: [ 2 34 34], output: [16 16 16]) - layer name: '0'
-------------------------------------------------------
	Weight shape: (16, 2, 5, 5)
	Stride: [2 2], Padding: [1 1], Dilation: [1 1]
	Groups: 1, Bias shape: (16,)
	Is recurrent: NO
	Dependencies:
	   - input (ready)

-------------------------------------------------------
IF (input: [16 16 16], output: [16 16 16]) - layer name: '1'
-------------------------------------------------------
	Parameter shape: (16, 16, 16)
	r range: [1.0000, 1.0000]
	v_threshold range: [1.0000, 1.0000]
	v_reset range: [0.0000, 0.0000]
	Is recurrent: NO
	Dependencies:
	   - 0 (ready)

-------------------------------------------------------
Conv2d (input: [16 16 16], output: [16 16 16]) - layer name: '2'
---

**OBS:** Para debugar o modelo, é melhor usar float (mais rápido e sem erro de quantização).

In [21]:
neuro_hls.implement_model(model, use_float=False)

## Criando o Testbench

In [22]:
neuro_hls.define_test_dataset("nir_examples/cnn_teste.npz", data_is_binary=True, step_count=300, different_sample_per_step=True)

In [25]:
neuro_hls.create_testbench(total_samples=100, batch_size=2, reset_potentials=True, debug_mode=False)

Total samples used: 100 of 1024
Batch size: 2
Total batches: 50
Testbench was created.


**OBS:** Se a simulação misteriosamente não rodar, considere diminuir o `batch_size` do `create_testbench`.

**OBS 2:** Para a rede convolucional, é preciso zerar os potenciais entre inferências. Por enquanto, fazer manualmente.

In [26]:
neuro_hls.run_csim()


****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2024.2 (64-bit)
  **** SW Build 5238294 on Nov  8 2024
  **** IP Build 5239520 on Sun Nov 10 16:12:51 MST 2024
  **** SharedData Build 5239561 on Fri Nov 08 14:39:27 MST 2024
  **** Start of session at: Thu Mar 19 15:17:28 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2024 Advanced Micro Devices, Inc. All Rights Reserved.

source /tools/Xilinx/Vitis/2024.2/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] For user 'user3' on host 'user3-Z690-PG-Riptide' (Linux_x86_64 version 6.11.0-28-generic) on Thu Mar 19 15:17:28 -03 2026
INFO: [HLS 200-10] On os Ubuntu 24.04.3 LTS
INFO: [HLS 200-10] In directory '/home/user3/Documentos/Fernando/NeuroHLS_dev_nir_to_cpp/NeuroHLS_dev/z_test_18_mar'
Sourcing Tcl script '1_csim.tcl'
INFO: [HLS 200-1510] Running: source 1_csim.tcl
INFO: [HLS 200-1510] Running: open_project vitis_proj 
INFO: [HLS 200-10] Opening project '/home/user3/Documentos/Fe

In [ ]:
neuro_hls.run_synth(frequency_MHz=200)

In [9]:
performance_estimates = neuro_hls.get_synth_performance_estimates()
print(performance_estimates)

{'avg_total_cycles': 1790497}


In [10]:
resource_usage = neuro_hls.get_synth_resource_usage()
print(resource_usage)

{'BRAM_18K': 1, 'DSP': 1, 'FF': 251, 'LUT': 552, 'URAM': 0}
